In [1]:
import json
import networkx as nx
import pandas as pd

# Ajusta esta ruta al fichero real
PATH = 'MC1_graph.json'
with open(PATH, 'r') as f:
    raw = json.load(f)

G = nx.node_link_graph(raw, directed=True, multigraph=True)


In [2]:
import json
import networkx as nx
import pandas as pd

# Parámetros de scoring
PTS_PER_RELEASE = 4.0      # puntos por canción/álbum lanzado
PTS_PER_HIT     = 15.0      # puntos extra por cada obra que chartea
DECAY_RATE      = 0.02      # % que pierde el score si no hay nuevo aporte

# Pesos de influencia por tipo de arista
WEIGHTS = {
    'InStyleOf':         1.0,
    'DirectlySamples':   1.5,
    'CoverOf':           1.2,
    'InterpolatesFrom':  1.3,
    'LyricalReferenceTo':0.8
}
# Decay para influencia
DECAY_INF = 0.1           # % de decay anual en influencia

# Roles que califican como obra del artista
OWN_ROLES = {'PerformerOf', 'ComposerOf', 'LyricistOf', 'ProducerOf'}

# 1. Carga el grafo
with open('MC1_graph.json') as f:
    G = nx.node_link_graph(json.load(f), directed=True, multigraph=True)

# 2. Detecta actores: Person artistas y MusicalGroup
ARTIST_ROLES = {'PerformerOf','MemberOf'}
actors = []
for n, attrs in G.nodes(data=True):
    if attrs.get('Node Type') == 'Person':
        if any(e['Edge Type'] in ARTIST_ROLES for _,_,e in G.out_edges(n,data=True)):
            actors.append((n, 'Person'))
    elif attrs.get('Node Type') == 'MusicalGroup':
        actors.append((n, 'Group'))

# 3. Determina rango de años
years = []
for _,attrs in G.nodes(data=True):
    for k in ('release_date','notoriety_date'):
        v = attrs.get(k)
        if isinstance(v,str) and v.isdigit():
            years.append(int(v))
min_year, max_year = (min(years), max(years)) if years else (0,0)

# 4. Construye scores año a año
records = []
for aid, atype in actors:
    pop_score = 0.0
    inf_score = 0.0
    for y in range(min_year, max_year+1):
        # Definir obras según tipo, ahora con OWN_ROLES
        if atype == 'Person':
            own = [
                tgt for _, tgt, e in G.out_edges(aid, data=True)
                if e['Edge Type'] in OWN_ROLES
                and G.nodes[tgt].get('Node Type') in ('Song','Album')
            ]
            bands = [m for _,m,e in G.out_edges(aid,data=True) if e['Edge Type']=='MemberOf']
            group_works = []
            for b in bands:
                group_works += [
                    tgt for _, tgt, e in G.out_edges(b, data=True)
                    if G.nodes[tgt].get('Node Type') in ('Song','Album')
                ]
            works = set(own) | set(group_works)
        else:  # 'Group'
            works = {
                tgt for _, tgt, e in G.out_edges(aid, data=True)
                if e['Edge Type'] in OWN_ROLES
                and G.nodes[tgt].get('Node Type') in ('Song','Album')
            }

        # Popularidad: releases y hits este año
        rels = sum(
            1 for w in works
            if G.nodes[w].get('release_date','').isdigit()
            and int(G.nodes[w]['release_date']) == y
        )
        hits = sum(
            1 for w in works
            if G.nodes[w].get('notable') == True
            and G.nodes[w].get('notoriety_date','').isdigit()
            and int(G.nodes[w]['notoriety_date']) == y
        )

        # Aportes anuales
        pop_gain = rels * PTS_PER_RELEASE + hits * PTS_PER_HIT
        pop_score = pop_score * (1 - DECAY_RATE) + pop_gain

        # Influencia: aristas de estilo
        inf_gain = 0.0
        for w in works:
            for _, tgt, e in G.out_edges(w, data=True):
                et = e['Edge Type']
                if et in WEIGHTS:
                    rd = G.nodes[tgt].get('release_date')
                    if rd and rd.isdigit() and int(rd) <= y:
                        inf_gain += WEIGHTS[et]
        inf_score = inf_score * (1 - DECAY_INF) + inf_gain

        records.append({
            'actor_id':  aid,
            'type':      atype,
            'year':      y,
            'releases':  rels,
            'hits':      hits,
            'pop_gain':  pop_gain,
            'pop_score': pop_score,
            'inf_gain':  inf_gain,
            'inf_score': inf_score
        })

# 5. Crear DataFrame y guardar
df = pd.DataFrame(records)
df.to_csv('actor_scores2.csv', index=False)
print("Generado actor_scores.csv con OWN_ROLES extendidos")


Generado actor_scores.csv con OWN_ROLES extendidos


In [ ]:
import json
import networkx as nx
import pandas as pd

# Ruta al JSON
GRAPH_JSON_PATH = 'MC1_graph.json'
with open(GRAPH_JSON_PATH, 'r') as f:
    G = nx.node_link_graph(json.load(f), directed=True, multigraph=True)

# Roles para identificar artistas y bandas
ARTIST_ROLES = {'PerformerOf', 'MemberOf'}

# Diccionario para mapear cada actor (Person o MusicalGroup) a su género
genre_map = {}

# 1) Género para cada Person que sea artista
for node, attrs in G.nodes(data=True):
    if attrs.get('Node Type') != 'Person':
        continue
    # Verificar si es artista
    roles = {e['Edge Type'] for _, _, e in G.out_edges(node, data=True)}
    if not (roles & ARTIST_ROLES):
        continue

    # Obras propias del artista
    own_works = [
        tgt for _, tgt, e in G.out_edges(node, data=True)
        if e['Edge Type'] in ARTIST_ROLES
        and G.nodes[tgt].get('Node Type') in ('Song', 'Album')
    ]
    # Obras de bandas a las que pertenece
    group_ids = [
        tgt for _, tgt, e in G.out_edges(node, data=True)
        if e['Edge Type'] == 'MemberOf'
    ]
    group_works = []
    for gid in group_ids:
        group_works += [
            tgt for _, tgt, e in G.out_edges(gid, data=True)
            if G.nodes[tgt].get('Node Type') in ('Song', 'Album')
        ]
    all_works = set(own_works) | set(group_works)

    # Recolectar géneros de esas obras
    genres = [
        G.nodes[w]['genre'] for w in all_works
        if G.nodes[w].get('genre')
    ]
    if genres:
        genre_map[node] = pd.Series(genres).mode().iloc[0]
    else:
        genre_map[node] = None

# 2) Género para cada MusicalGroup
for node, attrs in G.nodes(data=True):
    if attrs.get('Node Type') != 'MusicalGroup':
        continue
    # Obras de la banda
    works = [
        tgt for _, tgt, e in G.out_edges(node, data=True)
        if G.nodes[tgt].get('Node Type') in ('Song', 'Album')
    ]
    genres = [
        G.nodes[w]['genre'] for w in works
        if G.nodes[w].get('genre')
    ]
    if genres:
        genre_map[node] = pd.Series(genres).mode().iloc[0]
    else:
        genre_map[node] = None

# 3) Convertir a DataFrame y guardar
genre_df = pd.DataFrame([
    {'actor_id': actor_id, 'genre': genre_map[actor_id]}
    for actor_id in genre_map
])
genre_df.to_csv('actor_genres.csv', index=False)

print("Géneros asignados guardados en actor_genres.csv")


Géneros asignados guardados en actor_genres.csv


In [3]:
import json
import pandas as pd

# Rutas a los archivos generados
ACTOR_SCORES_PATH = 'actor_scores2.csv'
ACTOR_GENRES_PATH = 'actor_genres.csv'
GRAPH_JSON_PATH   = 'MC1_graph.json'

# 1. Cargar los CSVs
scores_df = pd.read_csv(ACTOR_SCORES_PATH)
genres_df = pd.read_csv(ACTOR_GENRES_PATH)

# 2. Cargar nodos para mapear id -> nombre
with open(GRAPH_JSON_PATH, 'r') as f:
    data = json.load(f)
nodes = pd.DataFrame(data['nodes'])

# 3. Extraer un DataFrame de metadatos con id, nombre y tipo
meta = []
for _, row in nodes.iterrows():
    node_id = row['id']
    node_type = row.get('Node Type')
    name = row.get('name')
    stage_name = row.get('stage_name', '')
    display_name = stage_name if pd.notnull(stage_name) and stage_name else name
    if node_type in ['Person', 'MusicalGroup']:
        meta.append({
            'actor_id':    node_id,
            'type':        'Person' if node_type=='Person' else 'Group',
            'name':        display_name
        })
meta_df = pd.DataFrame(meta)

# 4. Merge de scores, géneros y metadatos
combined = scores_df.merge(genres_df, on='actor_id', how='left') \
                    .merge(meta_df, on=['actor_id','type'], how='left')

# 5. Mostrar resultados
combined.head(10)


,actor_id,type,year,releases,hits,pop_gain,pop_score,inf_gain,inf_score,genre,name
0,1,Person,1975,0,0,0.0,0.0,0.0,0.0,Oceanus Folk,Carlos Duffy
1,1,Person,1976,0,0,0.0,0.0,0.0,0.0,Oceanus Folk,Carlos Duffy
2,1,Person,1977,0,0,0.0,0.0,0.0,0.0,Oceanus Folk,Carlos Duffy
3,1,Person,1978,0,0,0.0,0.0,0.0,0.0,Oceanus Folk,Carlos Duffy
4,1,Person,1979,0,0,0.0,0.0,0.0,0.0,Oceanus Folk,Carlos Duffy
5,1,Person,1980,0,0,0.0,0.0,0.0,0.0,Oceanus Folk,Carlos Duffy
6,1,Person,1981,0,0,0.0,0.0,0.0,0.0,Oceanus Folk,Carlos Duffy
7,1,Person,1982,0,0,0.0,0.0,0.0,0.0,Oceanus Folk,Carlos Duffy
8,1,Person,1983,0,0,0.0,0.0,0.0,0.0,Oceanus Folk,Carlos Duffy
9,1,Person,1984,0,0,0.0,0.0,0.0,0.0,Oceanus Folk,Carlos Duffy


In [4]:
combined.to_csv('actor_data2.csv', index=False)
print("Guardado en actor_data.csv")


Guardado en actor_data.csv


In [5]:
import json
import pandas as pd

# Paths
SCORES_CSV = 'actor_data2.csv'
GENRES_CSV = 'actor_genres.csv'
GRAPH_JSON = 'MC1_graph.json'

# 1. Carga datos
scores_df = pd.read_csv(SCORES_CSV)
genres_df = pd.read_csv(GENRES_CSV)

# 2. Carga nodos para metadatos
with open(GRAPH_JSON, 'r') as f:
    data = json.load(f)
nodes_df = pd.DataFrame(data['nodes'])

# 3. Construye meta_df
meta = []
for _, row in nodes_df.iterrows():
    if row.get('Node Type') in ['Person','MusicalGroup']:
        actor_type = 'Person' if row['Node Type']=='Person' else 'Group'
        name = row.get('stage_name') or row.get('name')
        meta.append({'actor_id': row['id'], 'type': actor_type, 'name': name})
meta_df = pd.DataFrame(meta)

# 4. Merge y cálculos de deltas
df = scores_df.merge(genres_df, on='actor_id', how='left') \
              .merge(meta_df, on=['actor_id','type'], how='left')
df = df.sort_values(['actor_id','year'])
df['pop_delta'] = df.groupby('actor_id')['pop_score'].pct_change().fillna(0)*100
df['inf_delta'] = df.groupby('actor_id')['inf_score'].pct_change().fillna(0)*100

# 5. Guardar CSV final
df.to_csv('actor_data2.csv', index=False)
print("actor_data2.csv generado correctamente")


actor_data2.csv generado correctamente


In [10]:
import pandas as pd

# Ruta al CSV existente
ACTOR_DATA_PATH = 'actor_data.csv'

# 1. Carga el CSV maestro
df = pd.read_csv(ACTOR_DATA_PATH)

# 2. Calcula la métrica combinada: suma de popularidad e influencia
df['total_score'] = df['pop_score'] + df['inf_score']

# 3. Guarda el CSV actualizado
df.to_csv('actor_data_updated.csv', index=False)

print("actor_data_updated.csv generado con la columna total_score")


actor_data_updated.csv generado con la columna total_score


In [ ]:
def compute_total_by_zscore(df):
    df['pop_z'] = df.groupby('year')['pop_score'] \
                   .transform(lambda x: (x - x.mean())/x.std(ddof=0))
    df['inf_z'] = df.groupby('year')['inf_score'] \
                   .transform(lambda x: (x - x.mean())/x.std(ddof=0))
    # Total centrado en 0 (puede ser negativo); si prefieres >0, añade un offset
    df['total_score'] = df['pop_z'] + df['inf_z']
    return df

df = pd.read_csv('actor_data2.csv')
df = compute_total_by_zscore(df)
df.to_csv('actor_data_updated2.csv', index=False)


In [7]:
df = pd.read_csv('actor_data_updated2.csv')
# drop de genre_y,name_y
df = df.drop(columns=['genre_y', 'name_y'])

# Renombrar columnas para claridad
df = df.rename(columns={
    'genre_x': 'genre',
    'name_x': 'name'
})

# Guardar el DataFrame limpio
df.to_csv('actor_data_updated2.csv', index=False)


In [4]:
pip install prophet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 11.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [prophet]m4/5 [prophet]]
Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm

# 1. Carga datos históricos
df = pd.read_csv('actor_data_updated.csv')

# 2. Prepara la lista de años futuros que quieres predecir
FUTURE_YEARS = list(range(2041, 2046))
future_df = pd.DataFrame({'ds': FUTURE_YEARS})

# 3. Recorrer cada artista y ajustar dos modelos (pop e inf)
predictions = []

for aid, sub in tqdm(df.groupby('actor_id'), desc="Pronosticando actores"):
    # Necesitamos al menos 3 puntos históricos en cada serie
    pop = sub[['year','pop_score']].rename(columns={'year':'ds','pop_score':'y'}).sort_values('ds')
    inf = sub[['year','inf_score']].rename(columns={'year':'ds','inf_score':'y'}).sort_values('ds')
    if len(pop) < 3 or len(inf) < 3:
        continue

    # Modelo de popularidad
    m_pop = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
    m_pop.fit(pop)
    fc_pop = m_pop.predict(future_df)[['ds','yhat']].rename(columns={'yhat':'pop_pred'})

    # Modelo de influencia
    m_inf = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
    m_inf.fit(inf)
    fc_inf = m_inf.predict(future_df)[['ds','yhat']].rename(columns={'yhat':'inf_pred'})

    # Unir ambas predicciones
    fc = fc_pop.merge(fc_inf, on='ds')
    fc['actor_id'] = aid
    name = sub['name'].iloc[0]
    fc['name'] = name
    fc['genre'] = sub['genre'].iloc[0]

    # Calcular total proyectado
    fc['total_pred'] = fc['pop_pred'] + fc['inf_pred']

    # Añadir al listado general
    predictions.append(fc)

# 4. Concatenar y exportar
pred_df = pd.concat(predictions, ignore_index=True)
pred_df = pred_df[['actor_id','name','genre','ds','pop_pred','inf_pred','total_pred']]
pred_df.rename(columns={'ds':'year'}, inplace=True)

pred_df.to_csv('actor_predictions.csv', index=False)
print("Guardado actor_predictions.csv con predicciones 2041–2045 para cada artista.")


Pronosticando actores:   0%|          | 0/9401 [00:00<?, ?it/s]20:55:21 - cmdstanpy - INFO - Chain [1] start processing
20:55:21 - cmdstanpy - INFO - Chain [1] done processing
20:55:21 - cmdstanpy - INFO - Chain [1] start processing
20:55:21 - cmdstanpy - INFO - Chain [1] done processing
Pronosticando actores:   0%|          | 1/9401 [00:00<35:58,  4.36it/s]20:55:21 - cmdstanpy - INFO - Chain [1] start processing
20:55:21 - cmdstanpy - INFO - Chain [1] done processing
20:55:21 - cmdstanpy - INFO - Chain [1] start processing
20:55:21 - cmdstanpy - INFO - Chain [1] done processing
Pronosticando actores:   0%|          | 2/9401 [00:00<32:59,  4.75it/s]20:55:21 - cmdstanpy - INFO - Chain [1] start processing
20:55:21 - cmdstanpy - INFO - Chain [1] done processing
20:55:21 - cmdstanpy - INFO - Chain [1] start processing
20:55:21 - cmdstanpy - INFO - Chain [1] done processing
Pronosticando actores:   0%|          | 3/9401 [00:00<29:52,  5.24it/s]20:55:21 - cmdstanpy - INFO - Chain [1] start 

Guardado actor_predictions.csv con predicciones 2041–2045 para cada artista.


In [16]:
df = pd.read_csv('actor_predictions.csv')

# 2. Convierte la columna 'year' de formato 'YYYY-01-01' a entero YYYY
df['year'] = pd.to_datetime(df['year']).dt.year

# 3. Guarda el nuevo DataFrame
df.to_csv('actor_predictions.csv', index=False)

print("Nuevo CSV guardado como actor_predictions.csv")

Nuevo CSV guardado como actor_predictions.csv


In [17]:
df

,actor_id,name,genre,year,pop_pred,inf_pred,total_pred
0,1,Carlos Duffy,Oceanus Folk,2041,4.592022,23.893139,28.485162
1,1,Carlos Duffy,Oceanus Folk,2042,4.681587,24.484555,29.166142
2,1,Carlos Duffy,Oceanus Folk,2043,4.771152,25.075970,29.847121
3,1,Carlos Duffy,Oceanus Folk,2044,4.860716,25.667385,30.528101
4,1,Carlos Duffy,Oceanus Folk,2045,4.950526,26.260420,31.210947
...,...,...,...,...,...,...,...
47000,17409,Isla Quinn,Oceanus Folk,2041,7.074777,15.854736,22.929513
47001,17409,Isla Quinn,Oceanus Folk,2042,7.228020,16.127313,23.355333
47002,17409,Isla Quinn,Oceanus Folk,2043,7.381264,16.399890,23.781154
47003,17409,Isla Quinn,Oceanus Folk,2044,7.534507,16.672467,24.206974


In [10]:
import pandas as pd

# Cargar predicciones
df = pd.read_csv('actor_predictions_2.csv')

# Normalización por año usando z-score
def compute_total_pred_by_zscore(df):
    df['pop_z'] = df.groupby('year')['pop_pred'] \
                    .transform(lambda x: (x - x.mean()) / x.std(ddof=0))
    df['inf_z'] = df.groupby('year')['inf_pred'] \
                    .transform(lambda x: (x - x.mean()) / x.std(ddof=0))
    df['total_pred'] = df['pop_z'] + df['inf_z']
    return df

# Aplico normalización
df = compute_total_pred_by_zscore(df)

# Cargar metadatos para agregar nombre y género
meta = pd.read_csv('actor_data_updated2.csv')[['actor_id', 'name', 'genre']].drop_duplicates()

# Merge para añadir columnas deseadas
df = df.merge(meta, on='actor_id', how='left')

# Selección y orden final
df = df[['actor_id', 'name', 'genre', 'year', 'pop_pred', 'inf_pred', 'total_pred']]

# Guardar CSV listo para el gráfico
df.to_csv('actor_predictions_2.csv', index=False)
print("✅ Archivo listo guardado como actor_predictions_2.csv")


✅ Archivo listo guardado como actor_predictions_2.csv


In [59]:
import json
import networkx as nx

# Ajusta la ruta a tu JSON
GRAPH_JSON_PATH = 'MC1_graph.json'
TARGET_ID = 1716  # ID del artista o banda a inspeccionar

# Carga el grafo
with open(GRAPH_JSON_PATH, 'r') as f:
    G = nx.node_link_graph(json.load(f), directed=True, multigraph=True)

# Recorre las aristas salientes del actor
works = []
for _, tgt, e in G.out_edges(TARGET_ID, data=True):
    node = G.nodes[tgt]
    if node.get('Node Type') in ('Song', 'Album'):
        name = node.get('name')
        year = node.get('release_date') or node.get('notoriety_date') or 'Unknown'
        works.append((name, year))

# Imprime el listado
print(f"Obras de actor {TARGET_ID}:")
for name, year in works:
    print(f"- {name} ({year})")


Obras de actor 1716:
- Guardian of the Mountain Winds (2029)
- Guardian of the Mountain Winds (2029)
- Fragments of Joy (2023)
- Stones of the Holy City (2018)
- Stones of the Holy City (2018)
- Pilgrimage of Longing (2023)
- Tanah Kita Bangkit (2019)
- Radiant Silhouette (2026)
- Immovable Love (2026)
- Immovable Love (2026)
- Scarves and Streetlight Shadows (2016)
- Ethereal Afterthoughts (2018)
- Timelines Converge (2016)
- Timelines Converge (2016)
- Carols of the Silent Night (2017)
- Carols of the Silent Night (2017)
- Silent Hooves, Whispering Forest (2025)
- Silent Hooves, Whispering Forest (2025)
- Cheerful Pathways (2019)
- Maria's Obeskrivna Minnen (Maria's Unwritten Memories) (2023)
- Second String Soul (2028)
- Beethoven's Unheard Conversation (2024)
- Beethoven's Unheard Conversation (2024)
- Sovereign's Whisper (2025)
- Childhood Memories Never Fade (2023)
- Childhood Memories Never Fade (2023)
- Echoes of a Global Generation (2023)
- Soft Love's First Light (2023)
- Sof

In [37]:
import json
import networkx as nx
import pandas as pd

# 1. Ajusta esta ruta a tu JSON
GRAPH_JSON_PATH = 'MC1_graph.json'

# 2. Carga el grafo
with open(GRAPH_JSON_PATH, 'r') as f:
    G = nx.node_link_graph(json.load(f), directed=True, multigraph=True)

# 3. Identificar actores: solistas (Person con PerformerOf) y bandas (MusicalGroup)
ARTIST_ROLES = {'PerformerOf'}
actors = []
for n, attrs in G.nodes(data=True):
    if attrs.get('Node Type') == 'Person':
        if any(e['Edge Type'] in ARTIST_ROLES for _, _, e in G.out_edges(n, data=True)):
            actors.append((n, 'Person', attrs.get('name')))
    elif attrs.get('Node Type') == 'MusicalGroup':
        actors.append((n, 'Group', attrs.get('name')))

# 4. Contar obras (canciones o álbumes) interpretadas por cada actor
records = []
for actor_id, actor_type, actor_name in actors:
    count = sum(
        1 for _, tgt, e in G.out_edges(actor_id, data=True)
        if e['Edge Type'] == 'PerformerOf'
        and G.nodes[tgt]['Node Type'] in ('Song', 'Album')
    )
    records.append({
        'actor_id': actor_id,
        'type':     actor_type,
        'name':     actor_name,
        'num_works': count
    })

# 5. Mostrar los Top 10
df = pd.DataFrame(records)
top10 = df.sort_values('num_works', ascending=False).head(10)
print(top10.to_string(index=False))


 actor_id   type                name  num_works
    17255 Person        Sailor Shift         26
     1716 Person     Kimberly Snyder         22
    17167  Group     The Salty Wakes         21
      551 Person          Szymon Pyć         20
    17207  Group  Wayfinder's Lament         18
     2668 Person           Ping Tian         17
      405 Person Leyla Graf-Gotthard         16
     1952 Person            Yang Wan         16
    16744  Group        Quattro Voci         16
     2023 Person          Qiang Tang         16
